# M3 CLV 조건부 상품-lift 가중 그래프 — Dunnhumby

Historical CLV proxy `N_hat × V_hat`로 고객을 10개 분위로 나눈 뒤, 각 분위에서 상대적으로 많이 구매된 상품의 기존 user-item 엣지를 강화합니다. CLV는 그래프 엣지값에만 들어가며, 최종 점수는 일반 LightGCN dot product입니다.

M1·실제 CLV·N 분위 내 CLV-shuffle를 seed 42, 100 epoch에서 비교합니다. 최종 test·holdout은 생성하지 않는 사후적 과거 개발구간 탐색입니다.

## 1. Drive와 검토된 코드 준비

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess

drive.mount('/content/drive')
REVIEWED_SHA = '4f6b148518f8ed34c42d191558f03ce8600b1313'
REPO = Path('/content/clv-m2-lightgcn-runner')
os.chdir('/content')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', REVIEWED_SHA], check=True)
head = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert head == REVIEWED_SHA, (head, REVIEWED_SHA)
os.chdir(REPO)
print('reviewed source:', head)

## 2. M3 개입 위치·분할·학습 조건 확인

In [ ]:
import json, torch
from lightgcn_clv_lift_graph import configure_clv_lift_graph_run, preflight_summary

assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
cfg = configure_clv_lift_graph_run()
summary = preflight_summary(cfg)
assert summary['intervention_location'] == 'graph edge weights only'
assert summary['historical_development_split']['final_test_constructed'] is False
assert summary['historical_development_split']['holdout_constructed'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 3. M1·M3-CLV-lift·M3-shuffle 비교

M1은 동일 설정의 기존 결과를 재사용하고 M3 두 arm만 학습합니다. 중단되어도 저장된 epoch 다음부터 재개됩니다.

In [ ]:
from lightgcn_clv_lift_graph import run_clv_lift_graph

result = run_clv_lift_graph(cfg)
display(result)
print('\n균형 지표 판독:')
print(json.dumps(result.attrs['decision'], ensure_ascii=False, indent=2))
print('\n결과 파일:')
print(json.dumps(result.attrs['result_paths'], ensure_ascii=False, indent=2))